In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install transformers datasets evaluate rouge_score sentencepiece bert-score sacrebleu


Mounted at /content/drive
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 8.8 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=fc50ee757d5dd0b926f9ede0ea12c34d506d76bb9ddd495ec35ee7d7d35a35bd
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [2]:
import torch
from datasets import load_dataset
from transformers import MT5ForConditionalGeneration, MT5Tokenizer
import evaluate
import numpy as np


In [3]:
data_file = "/content/drive/MyDrive/combined_renumbered.ndjson"
dataset = load_dataset("json", data_files=data_file, split="train")

# Use same split logic as training
dataset = dataset.train_test_split(test_size=0.1, seed=42)
test_dataset = dataset["test"]

print("Test size:", len(test_dataset))


Generating train split: 0 examples [00:00, ? examples/s]

Test size: 2503


In [4]:
model_path = "/content/drive/MyDrive/mt5_checkpoints/checkpoint-16896"

tokenizer = MT5Tokenizer.from_pretrained(model_path)
model = MT5ForConditionalGeneration.from_pretrained(model_path)

device = "cpu"
model.to(device)
print("Using device:", device)


Using device: cpu


In [5]:
max_input_length = 256
max_target_length = 30

def preprocess_function(examples):
    inputs = [f"headline generation: {txt}" for txt in examples["text"]]
    model_inputs = tokenizer(inputs,
                             max_length=max_input_length,
                             padding="max_length",
                             truncation=True)

    labels = tokenizer(examples["headline"],
                       max_length=max_target_length,
                       padding="max_length",
                       truncation=True)

    model_inputs["labels"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in seq]
        for seq in labels["input_ids"]
    ]
    return model_inputs

tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=test_dataset.column_names
)


Map:   0%|          | 0/2503 [00:00<?, ? examples/s]

In [6]:
batch_size = 4
all_preds = []
all_labels = []

for i in range(0, len(tokenized_test), batch_size):
    batch = tokenized_test[i:i+batch_size]

    input_ids = torch.tensor(batch["input_ids"]).to(device)
    attention_mask = torch.tensor(batch["attention_mask"]).to(device)

    labels = torch.tensor(batch["labels"]).clone()
    labels[labels == -100] = tokenizer.pad_token_id

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=32,
            num_beams=5,
            length_penalty=1.2
        )

    decoded_preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    all_preds.extend(decoded_preds)
    all_labels.extend(decoded_labels)

print("Generation complete.")


Generation complete.


In [7]:
rouge = evaluate.load("rouge")

rouge_result = rouge.compute(predictions=all_preds, references=all_labels)

rouge_result = {k: round(v*100, 2) for k, v in rouge_result.items()}
print("\n=== ROUGE Scores (1,2,L) ===")
print(rouge_result)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



=== ROUGE Scores (1,2,L) ===
{'rouge1': np.float64(14.89), 'rouge2': np.float64(2.62), 'rougeL': np.float64(14.95), 'rougeLsum': np.float64(14.91)}


In [8]:
print("\n=== ROUGE Sample Predictions ===")
for i in range(5):
    print(f"\n--- Sample {i+1} ---")
    print("True:", all_labels[i])
    print("Pred:", all_preds[i])
    print("="*60)



=== ROUGE Sample Predictions ===

--- Sample 1 ---
True: این اے 2 سوات، آزاد امیدوار امجد علی خان 88 ہزار 938 ووٹ لے کر کامیاب
Pred: این اے 2 سوات کے غیر حتمی، غیر سرکاری نتیجے میں آزاد امیدوار امجد علی خان کامیاب

--- Sample 2 ---
True: ہائپر ٹینشن کے علاج میں مشکلات سے نمٹنے والی دوا کا کامیاب تجربہ
Pred: ہائپر ٹینشن کے علاج میں مشکلات کا سامنا کرنے والے مریضوں کیلئے مفید ثابت

--- Sample 3 ---
True: سید محمد صوفی کے انتقال پر پاکستانی اسپورٹس حلقوں میں سوگ
Pred: سینئر صحافی سید محمد صوفی کے انتقال سے پاکستان میں اسپورٹس حلقوں میں سوگ کی کیفیت طاری

--- Sample 4 ---
True: ایشیا کپ: بھارتی ٹیم نے اسپورٹس مین شپ نظر انداز کردی، میچ کے بعد پاکستانی کھلاڑی
Pred: ایشیا کپ: بھارتی ٹیم کی جانب سے اسپورٹس مین شپ کیخلاف اقدام سامنے آگیا

--- Sample 5 ---
True: ٹیکساس، ریپبلکن خاتون کی جانب سے قرآن کی توہین
Pred: امریکی ریاست ٹیکساس میں ریپبلکن جماعت کی امیدوار نے قرآن پاک کی توہین کی ناپاک جس


In [9]:
bleu_metric = evaluate.load("sacrebleu")

bleu_result = bleu_metric.compute(predictions=all_preds, references=[[r] for r in all_labels])
bleu_score = round(bleu_result["score"], 2)

print("\n=== BLEU Score ===")
print({"BLEU": bleu_score})



=== BLEU Score ===
{'BLEU': 31.4}


In [10]:
print("\n=== BLEU Sample Predictions ===")
for i in range(5):
    print(f"\n--- Sample {i+1} ---")
    print("Reference:", all_labels[i])
    print("Prediction:", all_preds[i])
    print("="*60)



=== BLEU Sample Predictions ===

--- Sample 1 ---
Reference: این اے 2 سوات، آزاد امیدوار امجد علی خان 88 ہزار 938 ووٹ لے کر کامیاب
Prediction: این اے 2 سوات کے غیر حتمی، غیر سرکاری نتیجے میں آزاد امیدوار امجد علی خان کامیاب

--- Sample 2 ---
Reference: ہائپر ٹینشن کے علاج میں مشکلات سے نمٹنے والی دوا کا کامیاب تجربہ
Prediction: ہائپر ٹینشن کے علاج میں مشکلات کا سامنا کرنے والے مریضوں کیلئے مفید ثابت

--- Sample 3 ---
Reference: سید محمد صوفی کے انتقال پر پاکستانی اسپورٹس حلقوں میں سوگ
Prediction: سینئر صحافی سید محمد صوفی کے انتقال سے پاکستان میں اسپورٹس حلقوں میں سوگ کی کیفیت طاری

--- Sample 4 ---
Reference: ایشیا کپ: بھارتی ٹیم نے اسپورٹس مین شپ نظر انداز کردی، میچ کے بعد پاکستانی کھلاڑی
Prediction: ایشیا کپ: بھارتی ٹیم کی جانب سے اسپورٹس مین شپ کیخلاف اقدام سامنے آگیا

--- Sample 5 ---
Reference: ٹیکساس، ریپبلکن خاتون کی جانب سے قرآن کی توہین
Prediction: امریکی ریاست ٹیکساس میں ریپبلکن جماعت کی امیدوار نے قرآن پاک کی توہین کی ناپاک جس


In [13]:
import numpy as np
import evaluate

bertscore_metric = evaluate.load("bertscore")

bertscore_result = bertscore_metric.compute(
    predictions=all_preds,
    references=all_labels,
    lang="ur"
)

bert_precision = round(np.mean(bertscore_result["precision"]) * 100, 2)
bert_recall    = round(np.mean(bertscore_result["recall"]) * 100, 2)
bert_f1        = round(np.mean(bertscore_result["f1"]) * 100, 2)

print("\n=== BERTScore ===")
print({
    "BERTScore_Precision": bert_precision,
    "BERTScore_Recall": bert_recall,
    "BERTScore_F1": bert_f1
})



=== BERTScore ===
{'BERTScore_Precision': np.float64(81.55), 'BERTScore_Recall': np.float64(82.02), 'BERTScore_F1': np.float64(81.75)}


In [14]:
print("\n=== BERTScore Sample Predictions ===")
for i in range(5):
    print(f"\n--- Sample {i+1} ---")
    print("Reference:", all_labels[i])
    print("Prediction:", all_preds[i])
    print("="*60)



=== BERTScore Sample Predictions ===

--- Sample 1 ---
Reference: این اے 2 سوات، آزاد امیدوار امجد علی خان 88 ہزار 938 ووٹ لے کر کامیاب
Prediction: این اے 2 سوات کے غیر حتمی، غیر سرکاری نتیجے میں آزاد امیدوار امجد علی خان کامیاب

--- Sample 2 ---
Reference: ہائپر ٹینشن کے علاج میں مشکلات سے نمٹنے والی دوا کا کامیاب تجربہ
Prediction: ہائپر ٹینشن کے علاج میں مشکلات کا سامنا کرنے والے مریضوں کیلئے مفید ثابت

--- Sample 3 ---
Reference: سید محمد صوفی کے انتقال پر پاکستانی اسپورٹس حلقوں میں سوگ
Prediction: سینئر صحافی سید محمد صوفی کے انتقال سے پاکستان میں اسپورٹس حلقوں میں سوگ کی کیفیت طاری

--- Sample 4 ---
Reference: ایشیا کپ: بھارتی ٹیم نے اسپورٹس مین شپ نظر انداز کردی، میچ کے بعد پاکستانی کھلاڑی
Prediction: ایشیا کپ: بھارتی ٹیم کی جانب سے اسپورٹس مین شپ کیخلاف اقدام سامنے آگیا

--- Sample 5 ---
Reference: ٹیکساس، ریپبلکن خاتون کی جانب سے قرآن کی توہین
Prediction: امریکی ریاست ٹیکساس میں ریپبلکن جماعت کی امیدوار نے قرآن پاک کی توہین کی ناپاک جس
